# Model Inference — Walmart Store Sales Forecasting

Loads the overall best model (**prophet_walmart_best**, wrapper v2) directly from
the MLflow Model Registry on Dagshub, predicts on the raw Kaggle **test set**,
and generates + submits `submission.csv`.

**Why Prophet as the final model:** across all trained architectures
(N-BEATS, DLinear, TFT, TimesFM zero-shot & LoRA fine-tuned, LightGBM, XGBoost,
Prophet), Prophet achieved the best Kaggle test score. Its structure — per-series
trend + yearly seasonality + explicit holiday effects, extrapolated over any
horizon in one shot with no recursion — matches the competition's 39-week
forecast task better than lag-feature models, whose short-horizon validation
scores did not transfer to the long test horizon.

**The registered pipeline runs directly on raw data:** the v2 wrapper takes raw
rows `[Store, Dept, Date, Weekly_Sales]` where history rows have real sales and
rows to forecast have `NaN` — it refits Prophet per series on the history and
predicts exactly the requested dates. No preprocessing needed in this notebook.

## 0. Data download (Kaggle)

In [ ]:
!pip install kaggle==1.5.16

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! chmod 600 /root/.kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip -o Walmart-Recruiting-Store-Sales-Forecasting.zip
! unzip -o train.csv.zip
! unzip -o test.csv.zip
! unzip -o sampleSubmission.csv.zip

## 1. Setup & Imports

In [ ]:
!pip install -q prophet mlflow dagshub

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import dagshub
import mlflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient

MODEL_NAME = 'prophet_walmart_best'

dagshub.init(repo_owner='ikvas22',
             repo_name='Walmart-Recruiting---Store-Sales-Forecasting',
             mlflow=True)

print('Setup OK')

## 2. Load Best Model from MLflow Model Registry

In [ ]:
client   = MlflowClient()
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest   = max(versions, key=lambda v: int(v.version))

print(f'Registered model : {MODEL_NAME}')
print(f'Latest version   : {latest.version}')
print(f'Source run       : {latest.run_id}')
print(f'Tags             : {latest.tags}')

model = mlflow.pyfunc.load_model(f'models:/{MODEL_NAME}/{latest.version}')
print('\nModel loaded from registry')

## 3. Predict on the Raw Test Set

The wrapper receives the concatenation of raw training history (real
`Weekly_Sales`) and raw test rows (`Weekly_Sales = NaN`). Per series it refits
Prophet on the history and forecasts exactly the requested NaN dates.

**Runtime note:** one Prophet fit per test series (~3,000 series) — expect
**10–25 minutes** on Colab CPU.

In [ ]:
train_raw = pd.read_csv('train.csv', parse_dates=['Date'])
test_raw  = pd.read_csv('test.csv',  parse_dates=['Date'])

print(f'Train : {train_raw.shape}  ({train_raw["Date"].min().date()} → {train_raw["Date"].max().date()})')
print(f'Test  : {test_raw.shape}  ({test_raw["Date"].min().date()} → {test_raw["Date"].max().date()})')

# History rows (real sales) + test rows (NaN sales = "predict these")
test_raw['Weekly_Sales'] = np.nan
model_input = pd.concat([
    train_raw[['Store', 'Dept', 'Date', 'Weekly_Sales']],
    test_raw[['Store', 'Dept', 'Date', 'Weekly_Sales']],
], ignore_index=True)

print(f'\nModel input: {len(model_input):,} rows ({test_raw.shape[0]:,} to predict)')

In [ ]:
import time
t0 = time.time()

df_pred = model.predict(model_input)

print(f'Done in {(time.time() - t0) / 60:.1f} min')
print(f'Predictions: {len(df_pred):,}')
print(f'Stats: min={df_pred["Weekly_Sales_pred"].min():,.2f}  '
      f'mean={df_pred["Weekly_Sales_pred"].mean():,.2f}  '
      f'max={df_pred["Weekly_Sales_pred"].max():,.2f}')

## 4. Generate Kaggle Submission

In [ ]:
submission = pd.DataFrame({
    'Id': (
        df_pred['Store'].astype(str) + '_' +
        df_pred['Dept'].astype(str)  + '_' +
        pd.to_datetime(df_pred['Date']).dt.strftime('%Y-%m-%d')
    ),
    'Weekly_Sales': df_pred['Weekly_Sales_pred'],
})

sample = pd.read_csv('sampleSubmission.csv')
print(f'Sample submission rows : {len(sample):,}')
print(f'Our submission rows    : {len(submission):,}')

# Align to the official Id order (Kaggle requires exact Id coverage)
submission = sample[['Id']].merge(submission, on='Id', how='left')

missing = submission['Weekly_Sales'].isna().sum()
if missing > 0:
    print(f'WARNING: {missing} Ids had no prediction — filling with 0')
    submission['Weekly_Sales'] = submission['Weekly_Sales'].fillna(0)

submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv written')
submission.head(10)

In [ ]:
! kaggle competitions submit -c Walmart-Recruiting-Store-Sales-Forecasting -f submission.csv -m "Prophet v2 (from MLflow Model Registry)"

In [ ]:
# Check the submission score
! kaggle competitions submissions -c Walmart-Recruiting-Store-Sales-Forecasting | head -5